In [45]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier


In [11]:
df = pd.read_csv("../data/WA_Fn-UseC_-Telco-Customer-Churn.csv")
print(df.shape)

(7043, 21)


In [12]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"].isna().sum()

np.int64(11)

In [13]:
x=df.drop(columns=["customerID", "Churn"])
y=df["Churn"].map({"Yes": 1, "No": 0})


In [14]:
print(x.shape)
print(y.value_counts())

(7043, 19)
Churn
0    5174
1    1869
Name: count, dtype: int64


In [15]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)
print(X_train.shape)
print(X_test.shape)

(5634, 19)
(1409, 19)


In [16]:
print(X_train["TotalCharges"].isna().sum())
print(X_test["TotalCharges"].isna().sum())

8
3


In [17]:
median_charges = X_train["TotalCharges"].median()

X_train["TotalCharges"] = X_train["TotalCharges"].fillna(median_charges)
X_test["TotalCharges"] = X_test["TotalCharges"].fillna(median_charges)

print("Train missing:", X_train["TotalCharges"].isna().sum())
print("Test missing:", X_test["TotalCharges"].isna().sum())

Train missing: 0
Test missing: 0


In [24]:
text_columns = X_train.select_dtypes(exclude="number").columns
text_columns

Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod'],
      dtype='str')

In [26]:
preprocessor = ColumnTransformer(
    transformers=[
        ("categories", OneHotEncoder(handle_unknown="ignore"), text_columns)
    ],
    remainder="passthrough"
)
X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

print("Encoded train shape:", X_train_encoded.shape)
print("Encoded test shape:", X_test_encoded.shape)

Encoded train shape: (5634, 45)
Encoded test shape: (1409, 45)


Baseline model

In [28]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train_encoded, y_train)

baseline_pred = baseline.predict(X_test_encoded)

print(f"Accuracy: {accuracy_score(y_test, baseline_pred):.1%}")
print(f"Churn recall: {recall_score(y_test, baseline_pred):.1%}")

Accuracy: 73.5%
Churn recall: 0.0%


logistic model


In [33]:
logistic_model = make_pipeline(
    StandardScaler(with_mean=False),
    LogisticRegression(max_iter=1000)
)
logistic_model.fit(X_train_encoded, y_train)
logistic_pred = logistic_model.predict(X_test_encoded)

print(f"Accuracy: {accuracy_score(y_test, logistic_pred):.1%}")
print(f"Churn recall: {recall_score(y_test, logistic_pred):.1%}")

Accuracy: 80.7%
Churn recall: 56.7%


Decision_tree model

In [42]:
tree_model = DecisionTreeClassifier(max_depth=10, random_state=42)
tree_model.fit(X_train_encoded, y_train)

tree_pred = tree_model.predict(X_test_encoded)

print(f"Accuracy: {accuracy_score(y_test, tree_pred):.1%}")
print(f"Churn recall: {recall_score(y_test, tree_pred):.1%}")

Accuracy: 75.3%
Churn recall: 52.7%


RandomForest

In [47]:

forest_model = RandomForestClassifier(n_estimators=100, random_state=42)
forest_model.fit(X_train_encoded, y_train)

forest_pred = forest_model.predict(X_test_encoded)

print(f"Accuracy: {accuracy_score(y_test, forest_pred):.1%}")
print(f"Churn recall: {recall_score(y_test, forest_pred):.1%}")


Accuracy: 78.1%
Churn recall: 48.4%
